# London Weather Forecasting

## Data Loading & Preprocessing

We have selected a London Weather Dataset that goes from 1979 to 2023. There are many columns which shows different weather variables such as, mean/max temperatures, precipitations, sun time, daily global radiation... We focused in daily mean temperatures. Therefore, we have to make some changes to adapt the dataset to our needs:
- The Date column: the format was like this "YYYYMMDD" so we need to change to the date format to ease out future work.
- The Temperatures: all temperatures were multiplied by 10 something common in this type of datasets, so we have to divide by 10.
- Empty data: in some days the mean temperature misses, so we decided to fill it with the "Forward fill" technique (It copies the temperature of the day before to complete the data).

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler

# --- 1. CARGA Y LIMPIEZA DE DATOS ---
print("Cargando el dataset de Londres...")

df = pd.read_csv('london_weather_data_1979_to_2023.csv')

# Arreglamos la fecha (convertimos 19790101 a formato fecha real)
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m%d')
df.set_index('DATE', inplace=True)

# Vamos a predecir 'TG' (Temperatura Media / Mean Temperature)
# Dividimos entre 10 porque el dataset original no usa decimales (23.0 significa 2.3ºC)
df['TG'] = df['TG'] / 10.0

# Rellenamos los 29 días vacíos usando la temperatura del día anterior
df['TG'] = df['TG'].ffill()

# Extraemos nuestra variable objetivo
temperaturas = df['TG'].values.reshape(-1, 1)

# Visualizamos la historia climática de Londres para tu reporte
plt.figure(figsize=(14, 5))
plt.plot(df.index, temperaturas, color='#d9534f', linewidth=0.5)
plt.title('Temperatura Media Diaria en Londres (1979 - 2023)', fontweight='bold', fontsize=14)
plt.ylabel('Temperatura (°C)')
plt.grid(True, alpha=0.3)
plt.show()

# --- 2. ESCALADO DE DATOS ---
# A las redes LSTM les encantan los datos entre -1 y 1
scaler = MinMaxScaler(feature_range=(-1, 1))
temp_escaladas = scaler.fit_transform(temperaturas)

# --- 3. CREACIÓN DE LAS VENTANAS DE TIEMPO (Sliding Windows) ---
def create_sequences(data, seq_length):
    xs = []
    ys = []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)] # Ej: Días del 1 al 30
        y = data[i + seq_length]     # Ej: Día 31 (el que queremos predecir)
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 30 # Miraremos los últimos 30 días para predecir el día siguiente
X, y = create_sequences(temp_escaladas, seq_length)

# Convertimos a tensores de PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

print(f"Forma de X (Entradas): {X_tensor.shape} -> (Total días, Ventana 30 días, 1 variable)")
print(f"Forma de y (Etiquetas a predecir): {y_tensor.shape}")

KeyboardInterrupt: 

### Exploratory Data Analysis (EDA)

Before feeding the data into our Deep Learning models, it is crucial to understand its underlying distribution and patterns. In this section, we analyze the overall temperature distribution and visualize the monthly seasonality. The clear seasonal cycles confirm that this is a classic Time-Series problem, making Recurrent Neural Networks (like LSTM and GRU) the ideal architectures to capture these sequential patterns.

In [2]:
import seaborn as sns

# --- Advanced Exploratory Data Analysis (EDA) ---
print("Generando gráficos de Análisis Exploratorio (EDA)...")

# Creamos una figura con 2 subgráficos (uno al lado del otro)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Histograma de distribución general
sns.histplot(df['TG'], bins=50, kde=True, color='#2ca02c', ax=axes[0])
axes[0].set_title('General Temperature Distribution in London', fontweight='bold')
axes[0].set_xlabel('Mean Temperature (°C)')
axes[0].set_ylabel('Frequency (Days)')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Boxplot por meses para ver la estacionalidad
# Creamos una columna temporal de 'Mes' solo para este gráfico
df_eda = df.copy()
df_eda['Month'] = df_eda.index.month

# Usamos una paleta de colores de frío a calor (coolwarm)
sns.boxplot(x='Month', y='TG', data=df_eda, palette='coolwarm', ax=axes[1])
axes[1].set_title('Seasonality: Temperature Spread by Month', fontweight='bold')
axes[1].set_xlabel('Month (1 = Jan, 12 = Dec)')
axes[1].set_ylabel('Mean Temperature (°C)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

### Long-Term Trend and Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# --- Long-Term Trend & Autocorrelation ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Media Móvil de 1 año (365 días) para ver la tendencia a largo plazo
df['Rolling_365'] = df['TG'].rolling(window=365).mean()
axes[0].plot(df.index, df['Rolling_365'], color='#8c564b', linewidth=2)
axes[0].set_title('Long-Term Trend (365-Day Rolling Average)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Average Temperature (°C)')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Autocorrelación (ACF) de los últimos 100 días
plot_acf(df['TG'].dropna(), lags=100, ax=axes[1], color='#1f77b4', alpha=0.05)
axes[1].set_title('Autocorrelation of Daily Temperatures (100 Lags)', fontweight='bold')
axes[1].set_xlabel('Lags (Days)')
axes[1].set_ylabel('Correlation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Chronological Train-Test Split

Unlike image classification where we can shuffle data randomly, time-series data must be split chronologically to prevent **data leakage** (using future data to predict the past). We will use the first 80% of the timeline (1979-2015) for training and the remaining 20% (2015-2023) to test our model's predictive power on unseen future dates.

In [14]:
# --- Chronological Split ---
train_size = int(len(X_tensor) * 0.8)

X_train = X_tensor[:train_size]
y_train = y_tensor[:train_size]
X_test = X_tensor[train_size:]
y_test = y_tensor[train_size:]

print(f"Training Data: {len(X_train)} days")
print(f"Testing Data (Unseen Future): {len(X_test)} days")

Training Data: 13124 days
Testing Data (Unseen Future): 3282 days


### LSTM Model Architecture

We define a Recurrent Neural Network (RNN) using an **LSTM (Long Short-Term Memory)** layer. LSTMs are specifically designed to capture sequential patterns over time by maintaining an internal state (memory), making them ideal for weather forecasting. Our model processes a sliding window of 30 days and outputs a single continuous value (the predicted temperature for day 31) through a fully connected linear layer.

In [15]:
import torch.nn as nn
import torch.optim as optim

# --- LSTM Architecture ---
class ClimaLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super(ClimaLSTM, self).__init__()
        
        # The LSTM layer (the memory of the model)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Final fully connected layer to output a single numerical value
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        # We only care about the network's state at the LAST day of the sequence
        out = out[:, -1, :] 
        out = self.fc(out)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ClimaLSTM().to(device)
print(f"Model successfully sent to: {device}")

Model successfully sent to: cpu


### Training Engine & DataLoader Setup

To prevent the LSTM from getting stuck in local minima, we must train the model using mini-batches rather than feeding the entire dataset at once. We use PyTorch's `DataLoader` with a batch size of 64. **Note:** While the train-test split must be strictly chronological, it is perfectly safe (and recommended) to shuffle the mini-batches during training, because the sequential integrity is already preserved inside each 30-day sliding window.

In [16]:
from torch.utils.data import TensorDataset, DataLoader

# --- Training Engine & DataLoader ---
# Create a PyTorch Dataset
train_dataset = TensorDataset(X_train, y_train)

# We use batch_size=64 and shuffle=True to improve generalization
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

criterion = nn.MSELoss()
# We lower the learning rate a bit to 0.001 for better convergence
optimizer = optim.Adam(model.parameters(), lr=0.001) 

print("Training Engine and DataLoader setup complete.")

Training Engine and DataLoader setup complete.


### Mini-Batch Training Loop

We iterate through the dataset for 50 epochs. In each epoch, the model processes the training data in smaller chunks (batches). This approach introduces slight noise into the gradient calculations, helping the optimizer to escape local minima and accurately capture the seasonal peaks and valleys of London's climate.

In [17]:
# --- Training Loop (Mini-Batches) ---
print("Starting LSTM training process with mini-batches...\n")
epochs = 50 

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    # We loop through the data in batches of 64
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calculate average loss for this epoch
    avg_loss = epoch_loss / len(train_loader)
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Average MSE Loss: {avg_loss:.4f}')

print("\nTraining process completed successfully!")

Starting LSTM training process with mini-batches...

Epoch [10/50], Average MSE Loss: 0.0094
Epoch [20/50], Average MSE Loss: 0.0094
Epoch [30/50], Average MSE Loss: 0.0094
Epoch [40/50], Average MSE Loss: 0.0094
Epoch [50/50], Average MSE Loss: 0.0094

Training process completed successfully!
